# Micrograd: автоматично диференциране от нулата

Karpathy's [micrograd](https://github.com/karpathy/micrograd) — scalar autograd engine in ~100 lines.
Три примера: прост израз → един неврон → обучаване на малка мрежа.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import random

In [ ]:
## Copy of Karpathy's micrograd

class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op # the op that produced this node, for graphviz / debugging / etc

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

In [ ]:
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Neuron(Module):

    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Layer(Module):

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

---
## Пример 1: прост израз  `f = (a + b) * c`

Всяка операция с `Value` изгражда граф. `.backward()` обхожда графа назад и попълва `.grad` за всеки листов възел.

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

f = (a + b) * c
f.backward()

print(f"f            = {f.data}")
print(f"∂f/∂a = {a.grad:.1f}   (очаквано: c       = {c.data:.1f})")
print(f"∂f/∂b = {b.grad:.1f}   (очаквано: c       = {c.data:.1f})")
print(f"∂f/∂c = {c.grad:.1f}   (очаквано: a+b     = {a.data + b.data:.1f})")

In [ ]:
a_t = torch.tensor(2.0,  requires_grad=True)
b_t = torch.tensor(-3.0, requires_grad=True)
c_t = torch.tensor(10.0, requires_grad=True)

((a_t + b_t) * c_t).backward()

print("PyTorch верификация:")
print(f"∂f/∂a = {a_t.grad.item():.1f}   ∂f/∂b = {b_t.grad.item():.1f}   ∂f/∂c = {c_t.grad.item():.1f}")

---
## Пример 2: един неврон  `o = relu(x₁w₁ + x₂w₂ + b)`

`relu(z) = max(0, z)` — активационната функция в текущата версия на micrograd.

In [ ]:
x1 = Value(2.0);   x2 = Value(1.0)
w1 = Value(1.0);   w2 = Value(-2.0)
b  = Value(3.0)

n = x1*w1 + x2*w2 + b    # = 2*1 + 1*(-2) + 3 = 3.0
o = n.relu()              # relu(3.0) = 3.0  →  ∂o/∂n = 1.0
o.backward()

print(f"n            = {n.data:.4f}")
print(f"o = relu(n)  = {o.data:.4f}")
print()
print(f"∂o/∂x1 = {x1.grad: .4f}    (очаквано:  w1 = {w1.data:.1f})")
print(f"∂o/∂x2 = {x2.grad: .4f}    (очаквано:  w2 = {w2.data:.1f})")
print(f"∂o/∂w1 = {w1.grad: .4f}    (очаквано:  x1 = {x1.data:.1f})")
print(f"∂o/∂w2 = {w2.grad: .4f}    (очаквано:  x2 = {x2.data:.1f})")
print(f"∂o/∂b  = {b.grad}    (очаквано:  1.0)")

In [ ]:
x1_t = torch.tensor(2.0,  requires_grad=True)
x2_t = torch.tensor(1.0,  requires_grad=True)
w1_t = torch.tensor(1.0,  requires_grad=True)
w2_t = torch.tensor(-2.0, requires_grad=True)
b_t  = torch.tensor(3.0,  requires_grad=True)

torch.relu(x1_t*w1_t + x2_t*w2_t + b_t).backward()

print("PyTorch верификация:")
print(f"∂o/∂x1 = {x1_t.grad.item(): .4f}    ∂o/∂w1 = {w1_t.grad.item(): .4f}")
print(f"∂o/∂x2 = {x2_t.grad.item(): .4f}    ∂o/∂w2 = {w2_t.grad.item(): .4f}")
print(f"∂o/∂b  = {b_t.grad.item()}")

---
## Пример 3: обучаваме малка мрежа

`MLP(2 → 8 → 1)` върху 2D двукласов набор — само micrograd, без PyTorch.

In [ ]:

np.random.seed(0)
X_pos = np.random.randn(25, 2) + np.array([ 1.5,  1.5])
X_neg = np.random.randn(25, 2) + np.array([-1.5, -1.5])
X = np.vstack([X_pos, X_neg]).tolist()
y = [1.0] * 25 + [-1.0] * 25

colors = ['#27ae60'] * 25 + ['#c0392b'] * 25
plt.figure(figsize=(5, 5))
plt.scatter([xi[0] for xi in X], [xi[1] for xi in X],
            c=colors, s=45, edgecolors='k', linewidth=0.5)
plt.title('Набор от данни  (25 + 25 точки, 2 класа)')
plt.xlabel('x₁'); plt.ylabel('x₂')
plt.tight_layout()
plt.show()

In [ ]:
random.seed(42)
model = MLP(2, [8, 1])
print(f"Параметри: {len(model.parameters())}")

losses = []
for step in range(100):
    ypred = [model(xi) for xi in X]
    loss = sum((yp - Value(yt))**2 for yp, yt in zip(ypred, y)) / len(y)

    model.zero_grad()
    loss.backward()

    for p in model.parameters():
        p.data -= 0.05 * p.grad

    losses.append(loss.data)
    if step % 20 == 0:
        print(f"Стъпка {step:3d}: loss = {loss.data:.4f}")

In [ ]:
x_lo = min(xi[0] for xi in X) - 0.5;  x_hi = max(xi[0] for xi in X) + 0.5
y_lo = min(xi[1] for xi in X) - 0.5;  y_hi = max(xi[1] for xi in X) + 0.5
gx, gy = np.meshgrid(np.linspace(x_lo, x_hi, 80), np.linspace(y_lo, y_hi, 80))
Z = np.array([model([float(a), float(b)]).data
              for a, b in zip(gx.ravel(), gy.ravel())]).reshape(gx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(losses, color='steelblue', linewidth=1.8)
axes[0].set_xlabel('Стъпка', fontsize=11)
axes[0].set_ylabel('MSE loss', fontsize=11)
axes[0].set_title('Намаляване на loss', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].contourf(gx, gy, Z, levels=30, cmap='RdYlGn', alpha=0.75)
axes[1].contour(gx, gy, Z, levels=[0], colors='black', linewidths=2)
axes[1].scatter([xi[0] for xi in X], [xi[1] for xi in X],
                c=colors, s=45, edgecolors='k', linewidth=0.5, zorder=5)
axes[1].set_title('Научена граница на решение  (micrograd MLP)', fontsize=12)
axes[1].set_xlabel('x₁', fontsize=11)
axes[1].set_ylabel('x₂', fontsize=11)

plt.tight_layout()
plt.show()